In [1]:
import pandas as pd
import numpy as np

dataset=pd.read_csv('student_dropout_dataset_v3.csv')
dataset

,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,4,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,5,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,23.9,Female,42286.0,No,4.62,92.0,0,10.0,Yes,Yes,5.5,1.60,0.99,0.97,Year 2,Arts,Bachelor,0
9996,9997,17.0,Female,61103.0,Yes,2.87,75.2,3,32.4,No,Yes,6.7,3.09,3.09,3.09,Year 1,Business,Master,1
9997,9998,19.4,Male,25000.0,Yes,4.73,74.9,4,25.4,No,No,3.5,3.45,3.37,3.43,Year 4,Business,Bachelor,0
9998,9999,22.1,Female,40302.0,Yes,5.85,74.2,1,5.0,No,Yes,6.2,3.35,3.34,3.34,Year 1,CS,High School,0


In [2]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Student_ID             10000 non-null  int64  
 1   Age                    10000 non-null  float64
 2   Gender                 10000 non-null  object 
 3   Family_Income          9500 non-null   float64
 4   Internet_Access        10000 non-null  object 
 5   Study_Hours_per_Day    9500 non-null   float64
 6   Attendance_Rate        10000 non-null  float64
 7   Assignment_Delay_Days  10000 non-null  int64  
 8   Travel_Time_Minutes    10000 non-null  float64
 9   Part_Time_Job          10000 non-null  object 
 10  Scholarship            10000 non-null  object 
 11  Stress_Index           9500 non-null   float64
 12  GPA                    10000 non-null  float64
 13  Semester_GPA           10000 non-null  float64
 14  CGPA                   10000 non-null  float64
 15  Sem

In [3]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Load your data (assuming it's named 'df')
# df = pd.read_csv('your_dataset.csv')

# 2. Identify Categorical and Numerical columns
categorical_cols = dataset.select_dtypes(include=['object']).columns
numerical_cols = dataset.select_dtypes(include=['int64', 'float64']).columns.drop('Student_ID')

In [4]:
categorical_cols

Index(['Gender', 'Internet_Access', 'Part_Time_Job', 'Scholarship', 'Semester',
       'Department', 'Parental_Education'],
      dtype='object')

In [5]:
numerical_cols

Index(['Age', 'Family_Income', 'Study_Hours_per_Day', 'Attendance_Rate',
       'Assignment_Delay_Days', 'Travel_Time_Minutes', 'Stress_Index', 'GPA',
       'Semester_GPA', 'CGPA', 'Dropout'],
      dtype='object')

In [6]:
# 3. Encode Categorical Data 
# Note: We keep NaNs as NaNs during encoding so the Imputer can see them
le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    # Only encode non-null values to preserve the 'missing' status for the imputer
    series = dataset[col]
    valid_mask = series.notnull()
    dataset.loc[valid_mask, col] = le.fit_transform(series[valid_mask])
    le_dict[col] = le

# 4. Scale the data
# Distance calculations fail if one variable has a much larger range than others
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(dataset.drop(columns=['Student_ID'])), 
                         columns=dataset.columns.drop('Student_ID'))

# 5. Apply KNN Imputer
# n_neighbors=5 is standard; weights='distance' gives closer neighbors more influence
imputer = KNNImputer(n_neighbors=5, weights='distance')
df_imputed_array = imputer.fit_transform(df_scaled)

# 6. Convert back to DataFrame and Inverse Scale
df_final = pd.DataFrame(df_imputed_array, columns=df_scaled.columns)
df_final = pd.DataFrame(scaler.inverse_transform(df_final), columns=df_final.columns)

# 7. Convert Categorical columns back to original labels (and round them)
for col in categorical_cols:
    df_final[col] = df_final[col].round().astype(int)
    # Map back to original strings if needed
    # df_final[col] = le_dict[col].inverse_transform(df_final[col])

print("Missing values after imputation:", df_final.isnull().sum().sum())

Missing values after imputation: 0


In [8]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    10000 non-null  float64
 1   Gender                 10000 non-null  int64  
 2   Family_Income          10000 non-null  float64
 3   Internet_Access        10000 non-null  int64  
 4   Study_Hours_per_Day    10000 non-null  float64
 5   Attendance_Rate        10000 non-null  float64
 6   Assignment_Delay_Days  10000 non-null  float64
 7   Travel_Time_Minutes    10000 non-null  float64
 8   Part_Time_Job          10000 non-null  int64  
 9   Scholarship            10000 non-null  int64  
 10  Stress_Index           10000 non-null  float64
 11  GPA                    10000 non-null  float64
 12  Semester_GPA           10000 non-null  float64
 13  CGPA                   10000 non-null  float64
 14  Semester               10000 non-null  int64  
 15  Dep